In [39]:
# RDKit Chemoinformatics SAR Workflow

#A reusable workflow for SDF validation, physicochemical descriptor calculation, Morgan fingerprint generation, Tanimoto similarity analysis, docking-score integration, and exploratory SAR-pair identification.

#**Inputs:** SDF compound library and docking-score CSV  
#**Outputs:** Compound-level SAR results and flagged SAR pairs

In [21]:
import sys
print(sys.executable)

/opt/anaconda3/envs/RDKit_SAR_project/bin/python


In [22]:
from pathlib import Path

import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdFingerprintGenerator
from rdkit import DataStructs

In [23]:
def load_sdf(sdf_path):
    supplier = Chem.SDMolSupplier(str(sdf_path))

    valid_mols = []
    invalid_count = 0

    for mol in supplier:
        if mol is None:
            invalid_count += 1
            continue

        valid_mols.append(mol)

    print(f"Valid molecules: {len(valid_mols)}")
    print(f"Invalid molecules: {invalid_count}")

    return valid_mols

In [24]:
SDF_PATH = Path("Example_Data.sdf")

molecules = load_sdf(SDF_PATH)

Valid molecules: 1
Invalid molecules: 0


In [25]:
def get_compound_id(mol, index):
    if mol.HasProp("_Name") and mol.GetProp("_Name").strip():
        return mol.GetProp("_Name").strip()

    else:
        return f"Compound_{index + 1}"

In [26]:
def calculate_descriptors(mol):
    mol_weight = Descriptors.MolWt(mol)
    clogp = Crippen.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)

    descriptors = {
        "MolWt": mol_weight,
        "cLogP": clogp,
        "TPSA": tpsa,
        "HBD": hbd,
        "HBA": hba
    }

    return descriptors


for i, mol in enumerate(molecules):
    compound_id = get_compound_id(mol, i)
    desc = calculate_descriptors(mol)

    print(compound_id)
    print(desc)

Structure90
{'MolWt': 431.43800000000005, 'cLogP': 3.459400000000002, 'TPSA': 105.64999999999999, 'HBD': 4, 'HBA': 3}


In [27]:
records = []

for i, mol in enumerate(molecules):
    compound_id = get_compound_id(mol, i)
    desc = calculate_descriptors(mol)

    record = {
        "Compound_ID": compound_id,
        **desc
    }

    records.append(record)

df = pd.DataFrame(records)

df

,Compound_ID,MolWt,cLogP,TPSA,HBD,HBA
0,Structure90,431.438,3.4594,105.65,4,3


In [28]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

fingerprints = []

for mol in molecules:
    fp = morgan_generator.GetFingerprint(mol)
    fingerprints.append(fp)

print(f"Generated {len(fingerprints)} fingerprint(s)")

Generated 1 fingerprint(s)


In [29]:
REFERENCE_ID = "Structure90"

reference_index = df.index[df["Compound_ID"] == REFERENCE_ID][0]
reference_fp = fingerprints[reference_index]

similarities = []

for fp in fingerprints:
    similarity = DataStructs.TanimotoSimilarity(reference_fp, fp)
    similarities.append(similarity)

df["Tanimoto_to_reference"] = similarities

df

,Compound_ID,MolWt,cLogP,TPSA,HBD,HBA,Tanimoto_to_reference
0,Structure90,431.438,3.4594,105.65,4,3,1.0


In [31]:
DOCKING_CSV = Path("docking_scores.csv")

docking_df = pd.read_csv(DOCKING_CSV)

df = df.merge(
    docking_df,
    on="Compound_ID",
    how="left"
)

df

,Compound_ID,MolWt,cLogP,TPSA,HBD,HBA,Tanimoto_to_reference,Glide_XP
0,Structure90,431.438,3.4594,105.65,4,3,1.0,-8.5


In [32]:
def flag_sar_pairs(df, fingerprints, similarity_cutoff=0.7, score_diff_cutoff=1.0):
    sar_pairs = []

    for i in range(len(df)):
        for j in range(i + 1, len(df)):

            similarity = DataStructs.TanimotoSimilarity(
                fingerprints[i],
                fingerprints[j]
            )

            score_i = df.loc[i, "Glide_XP"]
            score_j = df.loc[j, "Glide_XP"]

            score_difference = abs(score_i - score_j)

            if similarity >= similarity_cutoff and score_difference >= score_diff_cutoff:
                sar_pairs.append({
                    "Compound_1": df.loc[i, "Compound_ID"],
                    "Compound_2": df.loc[j, "Compound_ID"],
                    "Tanimoto": similarity,
                    "XP_difference": score_difference
                })

    return pd.DataFrame(sar_pairs)

In [33]:
sar_pairs_df = flag_sar_pairs(
    df,
    fingerprints,
    similarity_cutoff=0.7,
    score_diff_cutoff=1.0
)

sar_pairs_df

""


In [34]:
OUTPUT_DIR = Path("SAR_results")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(
    OUTPUT_DIR / "compound_SAR_results.csv",
    index=False
)

sar_pairs_df.to_csv(
    OUTPUT_DIR / "flagged_SAR_pairs.csv",
    index=False
)

print(f"Results saved to: {OUTPUT_DIR.resolve()}")

Results saved to: /Users/aravind/SAR_results


In [35]:
def validate_inputs(df, reference_id):
    required_columns = ["Compound_ID", "Glide_XP"]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(f"Missing required column: {column}")

    if reference_id not in df["Compound_ID"].values:
        raise ValueError(f"Reference compound '{reference_id}' was not found.")

    if df["Glide_XP"].isna().any():
        print("Warning: Some compounds are missing Glide XP scores.")

    print("Input validation passed.")

In [36]:
validate_inputs(df, REFERENCE_ID)

Input validation passed.
